## 1) Load Silver Fact

In [1]:
import polars as pl
from pathlib import Path

SILVER_DIR = Path("../data/processed/silver")
GOLD_DIR = Path("../data/processed/gold")

GOLD_DIR.mkdir(parents=True, exist_ok=True)

fact = pl.read_parquet(SILVER_DIR / "fact_flights.parquet")

fact.shape

(3000000, 29)

## 2) Gold Mart: Airport Daily Performance

In [2]:
mart_airport_daily = (
    fact
    .group_by(["flight_date", "origin"])
    .agg([
        pl.len().alias("flights"),
        pl.mean("is_cancelled").alias("cancel_rate"),
        pl.mean("is_diverted").alias("divert_rate"),
        pl.mean("is_arr_delayed_15").alias("arr_delay_15_rate"),
        pl.mean("is_severe_arr_delay_60").alias("severe_delay_60_rate"),
        pl.mean("arr_delay_min").alias("avg_arr_delay_min"),
        pl.mean("taxi_out_min").alias("avg_taxi_out_min"),
    ])
    .rename({"origin": "airport"})
)

mart_airport_daily.write_parquet(GOLD_DIR / "mart_airport_daily.parquet")
mart_airport_daily.head()

flight_date,airport,flights,cancel_rate,divert_rate,arr_delay_15_rate,severe_delay_60_rate,avg_arr_delay_min,avg_taxi_out_min
date,str,u32,f64,f64,f64,f64,f32,f32
2021-12-26,"""LWS""",1,0.0,0.0,1.0,1.0,78.0,97.0
2022-03-01,"""BET""",1,0.0,0.0,0.0,0.0,-10.0,10.0
2021-05-31,"""RST""",1,0.0,0.0,0.0,0.0,-1.0,14.0
2021-02-15,"""BHM""",2,0.0,0.0,0.5,0.5,68.5,12.0
2020-03-30,"""SRQ""",4,0.75,0.0,0.0,0.0,-23.0,13.0


## 3) Gold Mart: Route Monthly Performance

In [3]:
mart_route_monthly = (
    fact
    .with_columns(
        pl.col("flight_date").dt.truncate("1mo").alias("month_start")
    )
    .group_by(["month_start", "origin", "dest"])
    .agg([
        pl.len().alias("flights"),
        pl.mean("is_cancelled").alias("cancel_rate"),
        pl.mean("is_arr_delayed_15").alias("arr_delay_15_rate"),
        pl.mean("is_severe_arr_delay_60").alias("severe_delay_60_rate"),
        pl.mean("arr_delay_min").alias("avg_arr_delay_min"),
        pl.mean("distance_miles").alias("avg_distance_miles"),
    ])
)

mart_route_monthly.write_parquet(GOLD_DIR / "mart_route_monthly.parquet")
mart_route_monthly.head()

month_start,origin,dest,flights,cancel_rate,arr_delay_15_rate,severe_delay_60_rate,avg_arr_delay_min,avg_distance_miles
date,str,str,u32,f64,f64,f64,f32,f32
2021-12-01,"""IAH""","""GUC""",2,0.5,0.5,0.0,45.0,886.0
2020-09-01,"""PHL""","""JAX""",2,0.0,0.0,0.0,-20.0,742.0
2023-01-01,"""ATL""","""LEX""",13,0.0,0.153846,0.0,-3.153846,304.0
2019-01-01,"""SDF""","""DTW""",9,0.0,0.222222,0.111111,8.444445,306.0
2022-11-01,"""MDW""","""SNA""",3,0.0,0.333333,0.0,15.0,1731.0


## 4) Gold Mart: Carrier Monthly Performance

In [4]:
mart_carrier_monthly = (
    fact
    .with_columns(
        pl.col("flight_date").dt.truncate("1mo").alias("month_start")
    )
    .group_by(["month_start", "carrier"])
    .agg([
        pl.len().alias("flights"),
        pl.mean("is_cancelled").alias("cancel_rate"),
        pl.mean("is_arr_delayed_15").alias("arr_delay_15_rate"),
        pl.mean("is_severe_arr_delay_60").alias("severe_delay_60_rate"),
        pl.mean("arr_delay_min").alias("avg_arr_delay_min"),
        pl.mean("taxi_out_min").alias("avg_taxi_out_min"),
    ])
)

mart_carrier_monthly.write_parquet(GOLD_DIR / "mart_carrier_monthly.parquet")
mart_carrier_monthly.head()

month_start,carrier,flights,cancel_rate,arr_delay_15_rate,severe_delay_60_rate,avg_arr_delay_min,avg_taxi_out_min
date,str,u32,f64,f64,f64,f32,f32
2021-01-01,"""OO""",5224,0.010528,0.094372,0.034265,-4.61828,17.942167
2021-08-01,"""DL""",7097,0.002395,0.134,0.041144,0.845424,15.273831
2022-01-01,"""9E""",2176,0.053309,0.154412,0.055607,2.23601,20.411451
2022-12-01,"""F9""",1449,0.05245,0.376812,0.138716,23.791393,18.38456
2019-07-01,"""G4""",1221,0.002457,0.210483,0.066339,8.309778,12.498358


## 5) Gold Mart: Delay Cause Analysis

In [5]:
cause_cols = [
    "delay_due_carrier",
    "delay_due_weather",
    "delay_due_nas",
    "delay_due_security",
    "delay_due_late_aircraft",
]

cause_cols = [c for c in cause_cols if c in fact.columns]

mart_delay_causes = (
    fact
    .filter((pl.col("is_arr_delayed_15") == 1) & (pl.col("is_cancelled") == 0))
    .with_columns(pl.col("flight_date").dt.truncate("1mo").alias("month_start"))
    .group_by(["month_start", "carrier"])
    .agg([pl.mean(c).alias(f"avg_{c}") for c in cause_cols])
)

mart_delay_causes.write_parquet(GOLD_DIR / "mart_delay_causes.parquet")
mart_delay_causes.head()

month_start,carrier,avg_delay_due_carrier,avg_delay_due_weather,avg_delay_due_nas,avg_delay_due_security,avg_delay_due_late_aircraft
date,str,f32,f32,f32,f32,f32
2023-04-01,"""MQ""",10.884727,7.305476,15.674352,0.610951,28.276657
2022-09-01,"""9E""",23.751055,2.725738,18.375528,0.0,30.286921
2020-01-01,"""B6""",28.892046,1.201705,13.471591,0.0,22.923296
2022-10-01,"""NK""",20.088472,2.927614,19.613941,0.297587,18.900805
2021-11-01,"""F9""",18.330854,0.211896,12.806691,0.0,29.639404
